# Official Replay Risk-State Target Screen v1

Goal: PK simple risk-state and volatility-stress target caps under the official warm-start replay path before any further PPO work.

External basis used for this narrow design:

- Qlib repository and reporting workflow: https://github.com/microsoft/qlib
- Qlib paper: https://arxiv.org/abs/2009.11189
- FinRL paper and repository for train/test/trade separation: https://arxiv.org/abs/2011.09607 and https://github.com/AI4Finance-Foundation/FinRL
- Backtest overfitting control: https://doi.org/10.21314/JCF.2016.322
- Volatility-managed portfolios: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431

Design note: `risk_target_position` is only used by the BC label builder when `is_downtrend_state=True`. Therefore this notebook includes a narrow control that only caps `risk_target_position`, plus variants that cap all target position columns on stress rows so the learned labels actually change.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.constrained_checkpoint_selector import (
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)
from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "official_replay_risk_state_target_screen_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
BASELINE_CARD = ROOT / "runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1"
TARGET_CARD = ROOT / "runtime/model_selection/card/short_volshock_target_generation_handoff_v1"
TARGETS_PATH = TARGET_CARD / "short_volshock_de_risk_targets.csv"
FRAME_PATH = ROOT / "data/soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
SHORT_VOLSHOCK_SLICE_ACCURACY_MIN = 0.60
SHORT_VOLSHOCK_SLICE_SAMPLES_MIN = 20
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
TARGET_POSITION_COLUMNS = ["teacher_position", "risk_target_position", "rebound_target_position"]

WALK_FORWARD_WINDOWS = {
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431",
]

CANDIDATES = [
    {
        "candidate": "bc_initialization",
        "mask": "none",
        "cap": math.nan,
        "columns": [],
        "description": "Original short-volshock BC target. Direct baseline.",
    },
    {
        "candidate": "risk_state_risk_target_cap010",
        "mask": "risk_state",
        "cap": 0.10,
        "columns": ["risk_target_position"],
        "description": "Control: cap risk_target_position only on existing risk_state rows.",
    },
    {
        "candidate": "risk_state_all_targets_cap010",
        "mask": "risk_state",
        "cap": 0.10,
        "columns": TARGET_POSITION_COLUMNS,
        "description": "Cap all BC target positions on existing risk_state rows.",
    },
    {
        "candidate": "volstress_all_targets_cap010",
        "mask": "volstress_fold_local",
        "cap": 0.10,
        "columns": TARGET_POSITION_COLUMNS,
        "description": "Cap all BC target positions on fold-local volatility/stress rows.",
    },
    {
        "candidate": "volstress_all_targets_cap005",
        "mask": "volstress_fold_local",
        "cap": 0.05,
        "columns": TARGET_POSITION_COLUMNS,
        "description": "Stronger cap on fold-local volatility/stress rows.",
    },
]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def bool_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame:
        return pd.Series(False, index=frame.index)
    return frame[column].astype(str).str.strip().str.lower().isin(["1", "true", "yes", "y"])


def numeric_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce")


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec["train"][1])
    subset = targets[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (pd.to_datetime(targets["datetime"]) <= train_end)
    ].copy()
    return subset.sort_values("datetime").reset_index(drop=True)


def gate_pass(relative_return: float, max_dd: float) -> bool:
    return bool(float(relative_return) >= TARGET_RELATIVE_RETURN and float(max_dd) <= TARGET_MAX_DD)


def gate_loss(relative_return: float, max_dd: float) -> float:
    return float(max(0.0, TARGET_RELATIVE_RETURN - float(relative_return)) + max(0.0, float(max_dd) - TARGET_MAX_DD) * 10.0)


def target_position_from_row(row: pd.Series) -> float:
    if bool_series(pd.DataFrame([row]), "is_downtrend_state").iloc[0]:
        return float(row.get("risk_target_position", np.nan))
    if bool_series(pd.DataFrame([row]), "is_rebound_cluster").iloc[0]:
        return float(row.get("rebound_target_position", np.nan))
    return float(row.get("teacher_position", np.nan))


def target_position_series(frame: pd.DataFrame) -> pd.Series:
    down = bool_series(frame, "is_downtrend_state")
    rebound = bool_series(frame, "is_rebound_cluster")
    risk = numeric_series(frame, "risk_target_position")
    rebound_pos = numeric_series(frame, "rebound_target_position")
    teacher = numeric_series(frame, "teacher_position")
    return pd.Series(np.where(down, risk, np.where(rebound, rebound_pos, teacher)), index=frame.index).clip(0.0, 1.0)


def fold_local_volstress_mask(fold_targets: pd.DataFrame) -> pd.Series:
    masks = []
    if "vix_percentile_252" in fold_targets:
        masks.append(numeric_series(fold_targets, "vix_percentile_252") >= 0.70)
    for col in ["regime_risk", "vol_of_vol_21", "realized_vol_21", "downside_semivol_21"]:
        values = numeric_series(fold_targets, col)
        finite = values.replace([np.inf, -np.inf], np.nan).dropna()
        if not finite.empty:
            masks.append(values >= float(finite.quantile(0.75)))
    masks.append(bool_series(fold_targets, "short_volshock_state"))
    if not masks:
        return pd.Series(False, index=fold_targets.index)
    result = masks[0].fillna(False).copy()
    for mask in masks[1:]:
        result = result | mask.fillna(False)
    return result.reindex(fold_targets.index).fillna(False)


def candidate_mask(fold_targets: pd.DataFrame, mask_name: str) -> pd.Series:
    if mask_name == "none":
        return pd.Series(False, index=fold_targets.index)
    if mask_name == "risk_state":
        return bool_series(fold_targets, "risk_state")
    if mask_name == "volstress_fold_local":
        return fold_local_volstress_mask(fold_targets)
    raise ValueError(f"unknown mask {mask_name}")


def apply_candidate(fold_targets: pd.DataFrame, candidate: dict[str, Any]) -> tuple[pd.DataFrame, dict[str, Any]]:
    modified = fold_targets.copy()
    before_target = target_position_series(modified)
    mask = candidate_mask(modified, str(candidate["mask"]))
    cap = candidate.get("cap", math.nan)
    columns = [col for col in candidate.get("columns", []) if col in modified.columns]
    if columns and math.isfinite(float(cap)):
        for col in columns:
            modified[col] = pd.to_numeric(modified[col], errors="coerce").clip(lower=0.0, upper=1.0)
            modified.loc[mask, col] = np.minimum(modified.loc[mask, col], float(cap))
    after_target = target_position_series(modified)
    changed_label = (before_target.round(8) != after_target.round(8)).fillna(False)
    audit = {
        "candidate": candidate["candidate"],
        "mask": candidate["mask"],
        "cap": cap,
        "columns": ",".join(columns),
        "rows": int(len(modified)),
        "mask_rows": int(mask.sum()),
        "target_label_changed_rows": int(changed_label.sum()),
        "target_label_mean_before": float(before_target.mean()),
        "target_label_mean_after": float(after_target.mean()),
        "target_label_delta_mean": float(after_target.mean() - before_target.mean()),
        "risk_state_rows": int(bool_series(modified, "risk_state").sum()),
        "downtrend_rows": int(bool_series(modified, "is_downtrend_state").sum()),
        "short_volshock_target_rows": int(bool_series(modified, "short_volshock_target_state").sum()),
    }
    for col in TARGET_POSITION_COLUMNS:
        if col in fold_targets:
            before = pd.to_numeric(fold_targets[col], errors="coerce")
            after = pd.to_numeric(modified[col], errors="coerce")
            audit[f"{col}_mean_before"] = float(before.mean())
            audit[f"{col}_mean_after"] = float(after.mean())
            audit[f"{col}_changed_rows"] = int((before.round(8) != after.round(8)).fillna(False).sum())
    return modified, audit


def max_drawdown_interval_from_daily(path: Path, *, candidate: str, fold: str) -> dict[str, Any]:
    daily = pd.read_csv(path, parse_dates=["datetime"])
    if daily.empty:
        return {"candidate": candidate, "fold": fold, "path": str(path.relative_to(ROOT)), "max_dd": math.nan}
    nav = pd.to_numeric(daily["strategy_nav"], errors="coerce").ffill()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    trough_idx = int(dd.idxmin())
    peak_idx = int(nav.loc[:trough_idx].idxmax())
    interval = daily.loc[peak_idx:trough_idx].copy()
    action_col = daily.get("action", pd.Series("", index=daily.index)).astype(str).str.upper()
    return {
        "candidate": candidate,
        "fold": fold,
        "path": str(path.relative_to(ROOT)),
        "peak_date": str(pd.Timestamp(daily.loc[peak_idx, "datetime"]).date()),
        "trough_date": str(pd.Timestamp(daily.loc[trough_idx, "datetime"]).date()),
        "max_dd": float(abs(dd.min()) * 100.0),
        "interval_days": int(len(interval)),
        "interval_avg_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).mean()),
        "interval_min_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).min()),
        "interval_max_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).max()),
        "interval_trades": int(action_col.loc[peak_idx:trough_idx].isin(["BUY", "SELL"]).sum()),
    }

profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
frame = pd.read_pickle(FRAME_PATH)

pd.DataFrame(CANDIDATES).to_csv(CARD / "candidate_configs.csv", index=False)
print({"experiment": EXPERIMENT, "profile": PROFILE_NAME, "frame_rows": int(len(frame)), "targets": int(len(targets)), "candidates": [c["candidate"] for c in CANDIDATES]})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'official_replay_risk_state_target_screen_v1', 'profile': 'sparse_release_bc_ppo_screen_v1', 'frame_rows': 2973, 'targets': 1009, 'candidates': ['bc_initialization', 'risk_state_risk_target_cap010', 'risk_state_all_targets_cap010', 'volstress_all_targets_cap010', 'volstress_all_targets_cap005']}


In [2]:

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    fold_targets = fold_subset(targets, spec)
    masks = {
        "risk_state": bool_series(fold_targets, "risk_state"),
        "short_volshock_state": bool_series(fold_targets, "short_volshock_state"),
        "short_volshock_target_state": bool_series(fold_targets, "short_volshock_target_state"),
        "is_downtrend_state": bool_series(fold_targets, "is_downtrend_state"),
        "volstress_fold_local": fold_local_volstress_mask(fold_targets),
    }
    row = {
        "fold": fold_name,
        "target_fold": spec["target_fold"],
        "target_rows": int(len(fold_targets)),
        "unique_target_dates": int(fold_targets["datetime"].nunique()) if len(fold_targets) else 0,
        "target_start": str(fold_targets["datetime"].min().date()) if len(fold_targets) else "",
        "target_end": str(fold_targets["datetime"].max().date()) if len(fold_targets) else "",
        "train_start": spec["train"][0],
        "train_end": spec["train"][1],
        "valid_start": spec["valid"][0],
        "valid_end": spec["valid"][1],
        "test_start": spec["test"][0],
        "test_end": spec["test"][1],
    }
    for name, mask in masks.items():
        row[f"{name}_rows"] = int(mask.sum())
        row[f"{name}_coverage"] = float(mask.mean()) if len(mask) else math.nan
    row["risk_and_downtrend_rows"] = int((masks["risk_state"] & masks["is_downtrend_state"]).sum())
    row["risk_not_downtrend_rows"] = int((masks["risk_state"] & ~masks["is_downtrend_state"]).sum())
    coverage_rows.append(row)

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / "coverage.csv", index=False)
write_json(
    CARD / "coverage.json",
    {
        "experiment": EXPERIMENT,
        "profile": PROFILE_NAME,
        "targets_path": str(TARGETS_PATH.relative_to(ROOT)),
        "feature_frame_path": str(FRAME_PATH.relative_to(ROOT)),
        "windows": coverage_rows,
        "skipped_windows": [
            {
                "fold": "wf_2020",
                "reason": "short-volshock targets begin in 2020, so a wf_2020 training target would leak that test year",
            }
        ],
        "external_sources": EXTERNAL_SOURCES,
    },
)
display(coverage)


,fold,target_fold,target_rows,unique_target_dates,target_start,target_end,train_start,train_end,valid_start,valid_end,...,short_volshock_state_rows,short_volshock_state_coverage,short_volshock_target_state_rows,short_volshock_target_state_coverage,is_downtrend_state_rows,is_downtrend_state_coverage,volstress_fold_local_rows,volstress_fold_local_coverage,risk_and_downtrend_rows,risk_not_downtrend_rows
0,wf_2022,wf_2022_train,253,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2021-01-01,2021-12-31,...,135,0.533597,133,0.525692,152,0.600791,152,0.600791,134,1
1,wf_2023_2024,wf_2023_2024_train,756,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2023-01-01,2023-12-31,...,342,0.452381,336,0.444444,508,0.671958,466,0.616402,342,0


In [3]:

init_qlib(QLIB_DATA_PATH)

fold_cache: dict[str, dict[str, Any]] = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {"dataset": dataset, "feature_full": feature_full, "price_full": price_full, "warm_seg": warm_seg}

warm_segments = pd.DataFrame(
    [
        {
            "fold": fold,
            "feature_start": str(cache["feature_full"].index.min().date()),
            "feature_end": str(cache["feature_full"].index.max().date()),
            "test_start": WALK_FORWARD_WINDOWS[fold]["test"][0],
            "test_end": WALK_FORWARD_WINDOWS[fold]["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "lookback": int(LOOKBACK),
        }
        for fold, cache in fold_cache.items()
    ]
)
warm_segments.to_csv(CARD / "warm_start_segments.csv", index=False)

official_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
daily_manifest: list[dict[str, str]] = []
interval_rows: list[dict[str, Any]] = []
target_patch_rows: list[dict[str, Any]] = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    raw_fold_targets = fold_subset(targets, spec)
    if raw_fold_targets.empty:
        for candidate in CANDIDATES:
            official_rows.append({"candidate": candidate["candidate"], "fold": fold_name, "status": "skipped_no_targets"})
        continue

    for candidate in CANDIDATES:
        candidate_name = str(candidate["candidate"])
        print(f"[screen] candidate={candidate_name} fold={fold_name}")
        fold_targets, audit = apply_candidate(raw_fold_targets, candidate)
        audit["fold"] = fold_name
        target_patch_rows.append(audit)

        set_seed(SEED)
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=BC_TRAIN_CONFIG,
            sample_weight_config=None,
            sample_size=0,
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=("short_volshock_target_state", "short_volshock_state", "risk_state", "is_downtrend_state", "is_rebound_cluster"),
        )
        slice_summary = summarize_bc_slice_fit(
            slice_metrics,
            required_slices=("short_volshock_target_state",),
            min_accuracy=SHORT_VOLSHOCK_SLICE_ACCURACY_MIN,
            min_samples=SHORT_VOLSHOCK_SLICE_SAMPLES_MIN,
        )
        tagged_slices = slice_metrics.copy()
        tagged_slices.insert(0, "fold", fold_name)
        tagged_slices.insert(0, "candidate", candidate_name)
        slice_frames.append(tagged_slices)
        dist = fit.action_distribution.copy()
        dist.insert(0, "fold", fold_name)
        dist.insert(0, "candidate", candidate_name)
        action_frames.append(dist)

        short_row = slice_metrics.set_index("slice").loc["short_volshock_target_state"].to_dict()
        fit_rows.append(
            {
                "candidate": candidate_name,
                "fold": fold_name,
                "target_rows": int(len(fold_targets)),
                "bc_samples": int(len(fit.actions)),
                "before_accuracy": float(fit.metrics.get("before_accuracy", math.nan)),
                "after_accuracy": float(fit.metrics.get("after_accuracy", math.nan)),
                "after_loss": float(fit.metrics.get("after_loss", math.nan)),
                "short_volshock_target_samples": int(short_row.get("samples", 0)),
                "short_volshock_target_accuracy": float(short_row.get("accuracy", math.nan)),
                "slice_gate": bool(slice_summary["slice_fit_gate"]),
                "slice_failed_checks": json.dumps(slice_summary.get("failed_checks", []), sort_keys=True),
            }
        )

        cache = fold_cache[fold_name]
        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = CARD / f"{candidate_name}_{fold_name}_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        daily_manifest.append({"candidate": candidate_name, "fold": fold_name, "path": str(daily_path.relative_to(ROOT))})
        interval_rows.append(max_drawdown_interval_from_daily(daily_path, candidate=candidate_name, fold=fold_name))
        metrics = daily_path_metrics(warm_daily)
        full_gate = gate_pass(float(metrics["relative_return"]), float(metrics["max_dd"]))
        official_rows.append(
            {
                "candidate": candidate_name,
                "fold": fold_name,
                "status": "evaluated",
                "requested_start": spec["test"][0],
                "requested_end": spec["test"][1],
                "warm_start": cache["warm_seg"][0],
                "warm_end": cache["warm_seg"][1],
                "official_warm_return": float(metrics["return"]),
                "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
                "official_warm_relative_return": float(metrics["relative_return"]),
                "official_warm_max_dd": float(metrics["max_dd"]),
                "official_warm_avg_pos": float(metrics["avg_pos"]),
                "trades": int(bt.get("trades", 0)),
                "slice_gate": bool(slice_summary["slice_fit_gate"]),
                "window_gate_pass": bool(full_gate and slice_summary["slice_fit_gate"]),
                "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])) + (0.0 if slice_summary["slice_fit_gate"] else 20.0),
                "daily_path": str(daily_path.relative_to(ROOT)),
            }
        )

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
bc_slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
official_metrics = pd.DataFrame(official_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
drawdown_intervals = pd.DataFrame(interval_rows).sort_values(["fold", "candidate"]).reset_index(drop=True) if interval_rows else pd.DataFrame()
target_patch_audit = pd.DataFrame(target_patch_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)

fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
bc_slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
official_metrics.to_csv(CARD / "official_replay_metrics.csv", index=False)
target_patch_audit.to_csv(CARD / "target_patch_audit.csv", index=False)
pd.DataFrame(daily_manifest).to_csv(CARD / "daily_manifest.csv", index=False)
drawdown_intervals.to_csv(CARD / "drawdown_interval_diagnostic.csv", index=False)

display(warm_segments)
display(target_patch_audit[["candidate", "fold", "mask_rows", "target_label_changed_rows", "target_label_mean_before", "target_label_mean_after"]])
display(fit_metrics[["candidate", "fold", "after_accuracy", "short_volshock_target_accuracy", "slice_gate"]])
display(official_metrics[["candidate", "fold", "official_warm_relative_return", "official_warm_max_dd", "official_warm_avg_pos", "window_gate_pass", "gate_loss"]])


[74592:MainThread](2026-05-27 02:13:16,565) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[74592:MainThread](2026-05-27 02:13:17,348) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[74592:MainThread](2026-05-27 02:13:17,348) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[74592:MainThread](2026-05-27 02:13:19,499) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[74592:MainThread](2026-05-27 02:13:19,500) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,feature_start,feature_end,test_start,test_end,warm_start,warm_end,lookback
0,wf_2022,2011-03-11,2022-12-30,2022-01-01,2022-12-31,2021-10-07,2022-12-30,60
1,wf_2023_2024,2011-03-11,2024-12-31,2024-01-01,2024-12-31,2023-10-05,2024-12-31,60


,candidate,fold,mask_rows,target_label_changed_rows,target_label_mean_before,target_label_mean_after
0,bc_initialization,wf_2022,0,0,0.263551,0.263551
1,bc_initialization,wf_2023_2024,0,0,0.212027,0.212027
2,risk_state_all_targets_cap010,wf_2022,135,0,0.263551,0.263551
3,risk_state_all_targets_cap010,wf_2023_2024,342,0,0.212027,0.212027
4,risk_state_risk_target_cap010,wf_2022,135,0,0.263551,0.263551
5,risk_state_risk_target_cap010,wf_2023_2024,342,0,0.212027,0.212027
6,volstress_all_targets_cap005,wf_2022,152,19,0.263551,0.252149
7,volstress_all_targets_cap005,wf_2023_2024,466,130,0.212027,0.174697
8,volstress_all_targets_cap010,wf_2022,152,14,0.263551,0.255163
9,volstress_all_targets_cap010,wf_2023_2024,466,112,0.212027,0.182513


,candidate,fold,after_accuracy,short_volshock_target_accuracy,slice_gate
0,bc_initialization,wf_2022,0.924901,1.000000,True
1,bc_initialization,wf_2023_2024,0.772185,0.925373,True
2,risk_state_all_targets_cap010,wf_2022,0.924901,1.000000,True
3,risk_state_all_targets_cap010,wf_2023_2024,0.772185,0.925373,True
4,risk_state_risk_target_cap010,wf_2022,0.924901,1.000000,True
5,risk_state_risk_target_cap010,wf_2023_2024,0.772185,0.925373,True
6,volstress_all_targets_cap005,wf_2022,0.916996,1.000000,True
7,volstress_all_targets_cap005,wf_2023_2024,0.830464,0.937313,True
8,volstress_all_targets_cap010,wf_2022,0.916996,1.000000,True
9,volstress_all_targets_cap010,wf_2023_2024,0.822517,0.940299,True


,candidate,fold,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,window_gate_pass,gate_loss
0,bc_initialization,wf_2022,67.624396,22.261989,0.097005,False,72.619890
1,bc_initialization,wf_2023_2024,5.018899,16.693087,0.183114,False,41.911976
2,risk_state_all_targets_cap010,wf_2022,67.624396,22.261989,0.097005,False,72.619890
3,risk_state_all_targets_cap010,wf_2023_2024,5.018899,16.693087,0.183114,False,41.911976
4,risk_state_risk_target_cap010,wf_2022,67.624396,22.261989,0.097005,False,72.619890
5,risk_state_risk_target_cap010,wf_2023_2024,5.018899,16.693087,0.183114,False,41.911976
6,volstress_all_targets_cap005,wf_2022,80.668313,11.953268,0.081543,True,0.000000
7,volstress_all_targets_cap005,wf_2023_2024,-3.160067,16.184661,0.171251,False,45.006679
8,volstress_all_targets_cap010,wf_2022,80.668313,11.953268,0.081543,True,0.000000
9,volstress_all_targets_cap010,wf_2023_2024,2.002060,15.111239,0.147366,False,29.110326


In [4]:

# Candidate PK and constrained selection.
gate_metrics = official_metrics[official_metrics["status"].eq("evaluated")].rename(
    columns={"fold": "window", "official_warm_relative_return": "relative_return", "official_warm_max_dd": "max_dd"}
)[["candidate", "window", "relative_return", "max_dd"]]
constraint_scores = score_constrained_checkpoints(gate_metrics) if not gate_metrics.empty else pd.DataFrame()
constraint_summary = summarize_constrained_checkpoint_selection(constraint_scores)
constraint_scores.to_csv(CARD / "constrained_checkpoint_scores.csv", index=False)

candidate_pk = official_metrics[official_metrics["status"].eq("evaluated")].copy()
candidate_pk["relative_rank"] = candidate_pk.groupby("fold")["official_warm_relative_return"].rank(ascending=False, method="min")
candidate_pk["maxdd_rank"] = candidate_pk.groupby("fold")["official_warm_max_dd"].rank(ascending=True, method="min")
candidate_pk["gate_rank"] = candidate_pk.groupby("fold")["gate_loss"].rank(ascending=True, method="min")
candidate_pk = candidate_pk.sort_values(["fold", "gate_loss", "official_warm_max_dd", "official_warm_relative_return"], ascending=[True, True, True, False]).reset_index(drop=True)
candidate_pk.to_csv(CARD / "candidate_pk.csv", index=False)

summary_by_candidate = candidate_pk.groupby("candidate", as_index=False).agg(
    folds=("fold", "nunique"),
    pass_windows=("window_gate_pass", "sum"),
    min_relative_return=("official_warm_relative_return", "min"),
    max_max_dd=("official_warm_max_dd", "max"),
    mean_avg_pos=("official_warm_avg_pos", "mean"),
    total_gate_loss=("gate_loss", "sum"),
)
summary_by_candidate["all_window_gate"] = summary_by_candidate["pass_windows"].eq(len(WALK_FORWARD_WINDOWS))
summary_by_candidate = summary_by_candidate.sort_values(
    ["all_window_gate", "total_gate_loss", "max_max_dd", "min_relative_return"],
    ascending=[False, True, True, False],
).reset_index(drop=True)
summary_by_candidate.to_csv(CARD / "candidate_summary.csv", index=False)

best_candidate = str(summary_by_candidate.iloc[0]["candidate"]) if not summary_by_candidate.empty else ""
best_all_gate = bool(summary_by_candidate.iloc[0]["all_window_gate"]) if not summary_by_candidate.empty else False
baseline = summary_by_candidate[summary_by_candidate["candidate"].eq("bc_initialization")]
baseline_max_dd = float(baseline.iloc[0]["max_max_dd"]) if not baseline.empty else math.nan
best_max_dd = float(summary_by_candidate.iloc[0]["max_max_dd"]) if not summary_by_candidate.empty else math.nan
best_min_rel = float(summary_by_candidate.iloc[0]["min_relative_return"]) if not summary_by_candidate.empty else math.nan

if best_all_gate:
    decision = "risk_state_target_screen_found_official_replay_candidate_no_promotion"
    next_mechanism = "run full fixed-split and longer walk-forward replay for the selected target cap before retaining any actor"
elif math.isfinite(baseline_max_dd) and math.isfinite(best_max_dd) and best_max_dd < baseline_max_dd:
    decision = "risk_state_target_cap_reduces_drawdown_but_misses_gate_no_promotion"
    next_mechanism = "try softer risk-state checkpoint scoring or reward timing instead of hard target cap"
else:
    decision = "risk_state_target_cap_screen_failed_no_promotion"
    next_mechanism = "drop hard target caps and move to risk-state checkpoint selection or reward-level drawdown timing"

conclusion = []
for row in summary_by_candidate.to_dict("records"):
    conclusion.append(
        f"{row['candidate']}: pass_windows={int(row['pass_windows'])}/{len(WALK_FORWARD_WINDOWS)}, "
        f"min_rel={row['min_relative_return']:.2f}pp, max_dd={row['max_max_dd']:.2f}%, "
        f"mean_avg_pos={row['mean_avg_pos']:.3f}, total_gate_loss={row['total_gate_loss']:.2f}."
    )
for row in candidate_pk[candidate_pk["candidate"].eq(best_candidate)].sort_values("fold").to_dict("records"):
    conclusion.append(
        f"Best candidate {best_candidate} on {row['fold']}: rel={row['official_warm_relative_return']:.2f}pp, "
        f"MaxDD={row['official_warm_max_dd']:.2f}%, avg_pos={row['official_warm_avg_pos']:.3f}, "
        f"gate={bool(row['window_gate_pass'])}."
    )
conclusion.append("No promotion or actor retention: this notebook is a target-screen PK under official warm-start replay only.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1/bc_initialization",
    "changed_layer": ["model_selection", "target", "bc_warm_start"],
    "external_reference": EXTERNAL_SOURCES,
    "hypothesis": "High volatility/stress rows explain the official replay drawdown gap; capping the BC target on those rows should reduce official MaxDD without losing the 30pp relative-return gate.",
    "acceptance_criteria": {"all_windows_relative_return_pp_min": TARGET_RELATIVE_RETURN, "all_windows_max_dd_pct_max": TARGET_MAX_DD},
    "accepted_tradeoff": "Lower average position is acceptable only if every official replay window keeps >=30pp excess and <=15% MaxDD.",
    "failure_mode": "Hard target caps may reduce exposure during both bad and good convexity days, lowering relative return while not changing execution timing enough to control drawdown.",
    "closest_historical_success_case": "short_volshock_bc_initialization cleared ordinary source fold gates before official replay invalidated it.",
    "closest_historical_failure_case": "short_volshock_bc_initialization_official_replay_v1 showed the source gate flipped under official warm-start replay due MaxDD/relative-return gaps.",
    "duplicate_check": "This run changes the BC target rows before fitting and PKs risk-state/all-target/volstress caps; it is not another PPO or renamed threshold-only profile.",
    "candidate_count": int(len(CANDIDATES)),
    "selected_candidate": best_candidate,
    "best_all_window_gate": best_all_gate,
    "best_min_relative_return": best_min_rel,
    "best_max_dd": best_max_dd,
    "selector_summary": constraint_summary,
    "promotion_allowed": False,
    "actor_retention": "no actor bundle retained; rerun notebook to regenerate diagnostic BC actors",
    "promotion_blocker": "partial walk-forward official replay target screen only; no full fixed split, no wf_2020-safe construction, no retained actor",
    "next_mechanism": next_mechanism,
    "sidecars": [
        "candidate_configs.csv",
        "coverage.csv",
        "coverage.json",
        "warm_start_segments.csv",
        "target_patch_audit.csv",
        "bc_fit_metrics.csv",
        "bc_slice_metrics.csv",
        "bc_action_distribution.csv",
        "official_replay_metrics.csv",
        "candidate_pk.csv",
        "candidate_summary.csv",
        "daily_manifest.csv",
        "drawdown_interval_diagnostic.csv",
        "constrained_checkpoint_scores.csv",
        "selection.json",
        "summary.json",
        "*_official_daily_path.csv",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "candidate_summary": summary_by_candidate.to_dict("records"),
    "candidate_pk": candidate_pk.to_dict("records"),
    "target_patch_audit": target_patch_audit.to_dict("records"),
    "fit_metrics": fit_metrics.to_dict("records"),
    "selector_summary": constraint_summary,
    "drawdown_interval_diagnostic": drawdown_intervals.to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)

display(candidate_pk[["fold", "candidate", "official_warm_relative_return", "official_warm_max_dd", "official_warm_avg_pos", "window_gate_pass", "gate_loss", "gate_rank"]])
display(summary_by_candidate)
display(constraint_scores)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


,fold,candidate,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,window_gate_pass,gate_loss,gate_rank
0,wf_2022,volstress_all_targets_cap005,80.668313,11.953268,0.081543,True,0.000000,1.0
1,wf_2022,volstress_all_targets_cap010,80.668313,11.953268,0.081543,True,0.000000,1.0
2,wf_2022,bc_initialization,67.624396,22.261989,0.097005,False,72.619890,3.0
3,wf_2022,risk_state_all_targets_cap010,67.624396,22.261989,0.097005,False,72.619890,3.0
4,wf_2022,risk_state_risk_target_cap010,67.624396,22.261989,0.097005,False,72.619890,3.0
5,wf_2023_2024,volstress_all_targets_cap010,2.002060,15.111239,0.147366,False,29.110326,1.0
6,wf_2023_2024,bc_initialization,5.018899,16.693087,0.183114,False,41.911976,2.0
7,wf_2023_2024,risk_state_all_targets_cap010,5.018899,16.693087,0.183114,False,41.911976,2.0
8,wf_2023_2024,risk_state_risk_target_cap010,5.018899,16.693087,0.183114,False,41.911976,2.0
9,wf_2023_2024,volstress_all_targets_cap005,-3.160067,16.184661,0.171251,False,45.006679,5.0


,candidate,folds,pass_windows,min_relative_return,max_max_dd,mean_avg_pos,total_gate_loss,all_window_gate
0,volstress_all_targets_cap010,2,1,2.002060,15.111239,0.114454,29.110326,False
1,volstress_all_targets_cap005,2,1,-3.160067,16.184661,0.126397,45.006679,False
2,bc_initialization,2,0,5.018899,22.261989,0.140060,114.531866,False
3,risk_state_all_targets_cap010,2,0,5.018899,22.261989,0.140060,114.531866,False
4,risk_state_risk_target_cap010,2,0,5.018899,22.261989,0.140060,114.531866,False


,candidate,selector_gate_pass,decision,constrained_selector_loss,constraint_gap_loss,failed_window_loss,drift_loss,score_loss,window_count,gate_pass_count,...,min_relative_return,max_max_dd,relative_drift_gap,max_dd_drift_gap,alignment_loss,checkpoint_objective_loss,missing_target_ratio,coverage_gate,target_relative_return_pp,target_max_dd_pct
0,volstress_all_targets_cap010,False,constrained_checkpoint_no_promotion,129.110326,29.110326,100.0,0.0,0.0,2,1,...,2.002060,15.111239,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0
1,volstress_all_targets_cap005,False,constrained_checkpoint_no_promotion,145.006679,45.006679,100.0,0.0,0.0,2,1,...,-3.160067,16.184661,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0
2,bc_initialization,False,constrained_checkpoint_no_promotion,314.531866,114.531866,200.0,0.0,0.0,2,0,...,5.018899,22.261989,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0
3,risk_state_all_targets_cap010,False,constrained_checkpoint_no_promotion,314.531866,114.531866,200.0,0.0,0.0,2,0,...,5.018899,22.261989,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0
4,risk_state_risk_target_cap010,False,constrained_checkpoint_no_promotion,314.531866,114.531866,200.0,0.0,0.0,2,0,...,5.018899,22.261989,0.0,0.0,0.0,0.0,0.0,True,30.0,15.0


{
  "experiment": "official_replay_risk_state_target_screen_v1",
  "created_at": "2026-05-26T18:15:08.387875+00:00",
  "decision": "risk_state_target_cap_reduces_drawdown_but_misses_gate_no_promotion",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1/bc_initialization",
  "changed_layer": [
    "model_selection",
    "target",
    "bc_warm_start"
  ],
  "external_reference": [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431"
  ],
  "hypothesis": "High volatility/stress rows explain the official replay drawdown gap; capping the BC target on those rows should reduce official MaxDD withou

## Runtime Conclusion

Read `selection.json`, `summary.json`, `candidate_summary.csv`, and `candidate_pk.csv` in this card for the single-run conclusion. This notebook intentionally keeps the conclusion local to the runtime card and does not update the active workpad or global experiment landscape unless a later promotion decision needs it.
